# The shape of London parkrun (Post 2)
Sources: as in `parkrun_access.ipynb`, plus OS Open Greenspace (sites and functions) and `known_start_dates.csv` (first-event dates from non-results pages; source URL per row). Distances are straight-line metres in EPSG:27700.

## Setup

In [1]:
import sys; sys.path.insert(0, ".")
import pandas as pd
import london_parkrun as lp

boroughs = lp.load_boroughs()
events = lp.filter_london_5k(lp.load_events(), boroughs)
lsoa = lp.lsoa_access(lp.load_lsoa(), events)
print(len(events), "events")

65 events


## Q7 In what order did they appear?

In [2]:
print(events.sort_values("id")[["id", "short_name"]].to_string(index=False))
known = pd.read_csv("known_start_dates.csv")
table, rho = lp.id_order_check(events, known)
print(table.to_string(index=False)); print("Spearman rho (id vs first event):", round(rho, 3), "n =", len(table))

  id                 short_name
   1                 Bushy Park
   2           Wimbledon Common
   4              Richmond Park
  15              Bedfont Lakes
  19            Roundshaw Downs
  24              Finsbury Park
  27                    Bromley
  29                     Bexley
  31                     Harrow
  44                 Grovelands
  51            Hackney Marshes
  53                 Avery Hill
  56              Old Deer Park
  72            Hampstead Heath
  75                   Kingston
  76                  Brockwell
  99            Beckenham Place
 126            Highbury Fields
 154             Crystal Palace
 183                 Valentines
 184                   Oak Hill
 190                      Lloyd
 191                Gunnersbury
 218              South Norwood
 230             Wanstead Flats
 241                Riddlesdown
 254                     Pymmes
 275                    Raphael
 300                 Ally Pally
 302            Wormwood Scrubs
 309    

## Q8 What are they called?

In [3]:
print(lp.name_terms(events).to_string())

short_name
(place name only)    29
Park                  6
Hill                  4
Fields                3
Common                3
Palace                2
End                   1
Path                  1
Country               1
Ground                1
Hospital              1
Meadows               1
Rye                   1
Lodge                 1
Flats                 1
Scrubs                1
Pally                 1
Norwood               1
Place                 1
Heath                 1
Marshes               1
Downs                 1
Lakes                 1
Peninsula             1


## Q9 Extremes

In [4]:
x = lp.event_extremes(events); print(x)

{'most_isolated': 'Stockley Country', 'isolated_nn_m': 6215.691131388652, 'closest_pair': ('Crane Park', 'Hanworth'), 'closest_m': 1391.5660961387703, 'median_nn_m': 2814.9614712836687}


## Q10 The London alphabet

In [5]:
print("missing:", lp.missing_letters(events))
route = lp.alphabet_route(events)
print(len(route), "letters;", round(lp.path_length(events, route) / 1000, 1), "km")
print(" → ".join(f"{lp.first_letter(n)}: {n}" for n in route))

missing: ['E', 'J', 'Q', 'U', 'X', 'Y', 'Z']


19 letters; 88.3 km
I: Ingrebourne Hill → V: Valentines → L: Lordship Recreation Ground → A: Ally Pally → M: Mile End → P: Peckham Rye → D: Dulwich → B: Brockwell → T: Tooting Common → C: Clapham Common → F: Fulham Palace → W: Wimbledon Common → K: Kingston → R: Richmond Park → O: Old Deer Park → G: Gunnersbury → S: Southall → N: Northala Fields → H: Harrow


## Q11 Where should the next one go?

In [6]:
cands = lp.candidate_sites(lp.load_greenspace(boroughs), boroughs, events)
picks = lp.best_new_sites(cands, lsoa)
print(len(cands), "candidate sites")
print(picks.drop(columns="geometry").round(1).to_string(index=False))
print("people > 2 km today:", int(lsoa.loc[lsoa.dist_m > 2000, "population"].sum()))

193 candidate sites
                        site              function  area_ha  added_population
Paddington Recreation Ground         Playing Field     10.7            169366
               West Ham Park Public Park Or Garden     26.5            112177
               London Fields Public Park Or Garden     12.8            109479
                Holland Park Public Park Or Garden     21.0            101960
        King Edward VII Park Public Park Or Garden     10.5            101424
people > 2 km today: 4100786


## Extra checks quoted in the post
Distance from Charing Cross (OSGB 530047, 180385) by ID third; nearest neighbour of the most isolated events; one-word names.

In [7]:
import numpy as np

ev = events.sort_values("id").reset_index(drop=True)
ev["cx_km"] = np.hypot(ev.geometry.x - 530047, ev.geometry.y - 180385) / 1000
ev["third"] = pd.qcut(ev.index, 3, labels=["oldest", "middle", "newest"])
print(ev.groupby("third", observed=True).agg(n=("id", "size"), mean_km=("cx_km", "mean")).round(1).to_string())
print(ev.loc[ev.short_name.isin(["Battersea", "Ingrebourne Hill"]), ["short_name", "id", "cx_km"]].round(1).to_string(index=False))

xy = np.column_stack([ev.geometry.x, ev.geometry.y])
d = np.hypot(xy[:, None, 0] - xy[None, :, 0], xy[:, None, 1] - xy[None, :, 1])
np.fill_diagonal(d, np.inf)
ev["nn_m"], ev["nn_name"] = d.min(axis=1), ev.short_name.to_numpy()[d.argmin(axis=1)]
print(ev.nlargest(3, "nn_m")[["short_name", "nn_name", "nn_m"]].round(0).to_string(index=False))

one_word = ev.short_name[~ev.short_name.str.contains(" ")]
print(len(one_word), "one-word names:", ", ".join(sorted(one_word)))

         n  mean_km
third              
oldest  22     13.4
middle  21     13.3
newest  22     13.4


      short_name   id  cx_km
       Battersea 3557    3.9
Ingrebourne Hill 3590   22.4
      short_name          nn_name   nn_m
Stockley Country         Southall 6216.0
          Morden Wimbledon Common 5737.0
          Harrow      Canons Park 4682.0
29 one-word names: Barking, Battersea, Beckton, Bexley, Brockwell, Bromley, Burgess, Catford, Charlton, Dulwich, Gladstone, Grovelands, Gunnersbury, Hanworth, Harrow, Hoblingwell, Kingston, Lloyd, Morden, Orpington, Osterley, Pymmes, Raphael, Riddlesdown, Southall, Southwark, Sutcliffe, Valentines, Walthamstow
